#### 4. Build CellOracle GRN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys, shutil, importlib, glob
import celloracle as co
import gimmemotifs
import faulthandler
faulthandler.enable()
import gc
gc.collect()
import scanpy as sc
import anndata as ad
import muon as mu

from celloracle import motif_analysis as ma
from celloracle.utility import save_as_pickled_object
from tqdm.notebook import tqdm
%matplotlib inline

In [ ]:
base_GRN_promoter = pd.read_parquet("/home/fuyq/GRN/single_cell/celloracle/pbmc10k/result/cellorcale/BaseGRN_input_data_preparation/base_GRN_dataframe.parquet")
oracle = co.Oracle()
rna.X = rna.layers["raw_count"].copy()
oracle.import_anndata_as_raw_count(adata=rna,
                                   cluster_column_name="celltype",
                                   embedding_name="X_umap")
oracle.import_TF_data(TF_info_matrix=base_GRN_promoter)

In [ ]:
# PCA
oracle.perform_PCA()
plt.plot(np.cumsum(oracle.pca.explained_variance_ratio_)[:100])
n_comps = np.where(np.diff(np.diff(np.cumsum(oracle.pca.explained_variance_ratio_))>0.002))[0][0]
plt.axvline(n_comps, c="k")
plt.show()
print(n_comps)
n_comps = min(n_comps, 50)


In [ ]:
# KNN imputation
n_cell = oracle.adata.shape[0]
print(f"cell number is :{n_cell}")
k = int(0.025*n_cell)
print(f"Auto-selected k is :{k}")
oracle.knn_imputation(n_pca_dims=n_comps, k=k, balanced=True, b_sight=k*8,
                      b_maxl=k*4, n_jobs=4)


In [ ]:
# Infer GRN links per cell type
links = oracle.get_links(cluster_name_for_GRN_unit="celltype", alpha=10,
                         verbose_level=10)
all_cluster_links = []
for cluster_name in links.links_dict:
    df = links.links_dict[cluster_name].copy()
    df["cluster"] = cluster_name
    all_cluster_links.append(df)
combined_df = pd.concat(all_cluster_links)